# HEAD

- a
- b

In [ ]:
from loguru import logger
logger.info("Importing libraries")
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

logger.info("Loading datasets")
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
markdown
#VSC-109819e4
markdown
# HEAD

- a
- b
code
#VSC-4aed062d
python
from loguru import logger
logger.info("Importing libraries")
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import seaborn as sns
import matplotlib.pyplot as plt

logger.info("Loading datasets")
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)
logger.info("done")

batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break





device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

# Tracking lists for plotting
train_losses = []
train_steps = []
test_losses = []
test_accuracies = []
step = 0

def train(dataloader, model, loss_fn, optimizer):
    global step
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Record loss and step
        train_losses.append(loss.item())
        train_steps.append(step)
        step += 1

        if batch % 100 == 0:
            loss_val, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")


def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    test_losses.append(test_loss)
    test_accuracies.append(correct)
    print(f"Test Error: \n+ Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n+")
    return test_loss, correct

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")
logger.info("Training complete")

torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

classes = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')
code
#VSC-ebc00018
python
# Plot training and test loss using seaborn
sns.set_style("whitegrid")
plt.figure(figsize=(10,5))
# training loss over steps
sns.lineplot(x=train_steps, y=train_losses, label="Train loss")
# test loss per epoch (plot on secondary axis)
ax = plt.gca()
ax2 = ax.twinx()
if len(test_losses) > 0:
    sns.lineplot(x=list(range(1, len(test_losses)+1)), y=test_losses, marker="o", label="Test loss", ax=ax2, color="orange")
    ax2.set_ylabel("Test loss")
ax.set_xlabel("Training step (train) / Epoch (test)")
ax.set_ylabel("Train loss")
ax.set_title("Training and Test Loss")
ax.legend(loc="upper left")
if len(test_losses) > 0:
    ax2.legend(loc="upper right")
plt.show()

# Optional: plot test accuracy per epoch
if len(test_accuracies) > 0:
    plt.figure(figsize=(6,3))
    sns.lineplot(x=list(range(1, len(test_accuracies)+1)), y=[a*100 for a in test_accuracies], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Test Accuracy per Epoch")
    plt.show()

2025-12-31 13:15:18.503 | INFO     | __main__:<module>:2 - Importing libraries
2025-12-31 13:15:22.921 | INFO     | __main__:<module>:9 - Loading datasets
2025-12-31 13:15:23.047 | INFO     | __main__:<module>:25 - done


Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Epoch 1
-------------------------------
loss: 2.313666  [   64/60000]
loss: 2.311051  [ 6464/60000]
loss: 2.285526  [12864/60000]
loss: 2.297324  [19264/60000]
loss: 2.293199  [25664/60000]
loss: 2.287811  [32064/60000]
loss: 2.268025  [38464/60000]
loss: 2.282788  [44864/60000]
loss: 2.273734  [51264/60000]
loss: 2.251040  [57664/60000]
Test Error: 
 Accuracy: 30.1%, Avg loss: 2.261046 

Epoch 2
-------------------------------
loss: 2.265377  [   64/60000]
loss: 2.258499  [ 6464/60000]
loss: 2.249679  [12864/60000]
loss: 2.237651  [19264/60000]
loss: 2.246459  [

2025-12-31 13:15:59.780 | INFO     | __main__:<module>:112 - Training complete


Test Error: 
 Accuracy: 69.4%, Avg loss: 1.629023 

Done!
Saved PyTorch Model State to model.pth
Predicted: "7", Actual: "7"
